# Qwen3-0.6B NumPy implementation: inference checks

This notebook clones [Qwen3Implementations](https://github.com/seungjun-green/Qwen3Implementations), loads the real Qwen3-0.6B weights into the from-scratch NumPy model, and compares it against the official Hugging Face model with three checks:

1. **Shape check:** logits come out as `(N, L, 151936)`.
2. **Logits comparison:** max absolute difference vs. Hugging Face, and whether the argmax matches at every position.
3. **Greedy generation (temperature 0):** generated tokens vs. Hugging Face `generate(do_sample=False)`.

**Note on hardware:** the NumPy model runs on the **CPU** (NumPy doesn't use the GPU). The GPU is used for the Hugging Face reference model.

In [ ]:
!nvidia-smi

In [ ]:
# Qwen3 support needs transformers >= 4.51
!pip install -q -U "transformers>=4.51"

## 1. Clone the repo

In [ ]:
REPO_URL = "https://github.com/seungjun-green/Qwen3Implementations"
REPO_DIR = "/content/Qwen3Implementations"

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}
!ls -la {REPO_DIR}

## 2. Load the implementation from the repo

Some repo files have no `.py` extension and also run test code at the bottom (e.g. building a ~4.8 GB random model), so they can't be imported directly.
This cell parses each file and executes **only its class and function definitions**, skipping the top-level code. The model code itself runs exactly as written in the repo.

In [ ]:
import ast
import numpy as np
import torch
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForCausalLM

# shared namespace the repo code runs in (provides the libraries it uses)
ns = {"np": np, "torch": torch, "dataclass": dataclass, "AutoModelForCausalLM": AutoModelForCausalLM}

def load_definitions(path, ns):
    tree = ast.parse(open(path).read(), filename=path)
    keep = [node for node in tree.body if isinstance(node, (ast.ClassDef, ast.FunctionDef))]
    exec(compile(ast.Module(body=keep, type_ignores=[]), path, "exec"), ns)
    return [node.name for node in keep]

print("Qwen3:          ", load_definitions(f"{REPO_DIR}/Qwen3", ns))
print("RealQwen3Weight:", load_definitions(f"{REPO_DIR}/RealQwen3Weight", ns))

Qwen3Config      = ns["Qwen3Config"]
Qwen3            = ns["Qwen3"]
RealQwen3Weights = ns["RealQwen3Weights"]
generate         = ns["generate"]

# generate() calls model.reset_cache(); add it if the Qwen3 class doesn't define it yet
if not hasattr(Qwen3, "reset_cache"):
    def reset_cache(self):
        for layer in self.decoder_layers:
            layer.masked_gqa_attention.reset_cache()
    Qwen3.reset_cache = reset_cache
    print("\nadded Qwen3.reset_cache()")

## 3. Build both models

- **Mine:** the NumPy `Qwen3` with real weights loaded by `RealQwen3Weights` (CPU, float32 weights).
- **Reference:** the official Hugging Face Qwen3-0.6B on the GPU in float32, with TF32 disabled so its matmuls stay full float32 precision.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_NEW_TOKENS = 20
prompts = [
    "The capital of France is",
    "def fibonacci(n):",
    "Once upon a time",
]

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

cfg = Qwen3Config()
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

w = RealQwen3Weights(cfg, MODEL_NAME)
model = Qwen3(w.num_layers, w.WE, w.final_gamma, w.eps, w.output_matrix, w.decoder_layer_parms)

hf = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device)
hf.eval()

def encode(text):
    return tok(text, return_tensors="np")["input_ids"]          # (1, L)

results = {}

## Check 1: output shape

In [ ]:
ids = encode(prompts[0])
model.reset_cache()
logits = model(ids)

expected = (1, ids.shape[1], cfg.vocab_size)
print("input ids:", ids.shape, "| logits:", logits.shape, "| expected:", expected)
results["shape"] = logits.shape == expected
print("PASS" if results["shape"] else "FAIL")

## Check 2: logits vs. Hugging Face

Full prompt in one forward pass (no generation). Small floating-point differences are expected; a max abs diff around `1e-4`–`1e-3` with the same argmax everywhere means the implementation matches.

In [ ]:
check2 = []
for p in prompts:
    ids = encode(p)

    model.reset_cache()
    mine = model(ids)                                            # (1, L, V)

    with torch.no_grad():
        ref = hf(torch.tensor(ids, device=device)).logits.float().cpu().numpy()

    max_diff = float(np.abs(mine - ref).max())
    same_argmax = bool((mine.argmax(-1) == ref.argmax(-1)).all())
    check2.append(same_argmax)
    print(f"{p!r:30} max abs diff = {max_diff:.2e} | same argmax: {same_argmax}")

results["logits"] = all(check2)
print("\nPASS" if results["logits"] else "\nFAIL")

## Check 3: greedy generation (temperature 0) vs. Hugging Face

Uses the KV-cache `generate` from the repo. Only the overlapping part is compared, since one side may stop early at EOS.

In [ ]:
check3 = []
for p in prompts:
    ids = encode(p)
    P = ids.shape[1]

    mine = generate(model, ids, max_new_tokens=MAX_NEW_TOKENS, eos_id=tok.eos_token_id)[0]

    with torch.no_grad():
        ref = hf.generate(
            torch.tensor(ids, device=device),
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )[0].cpu().numpy()

    n = min(len(mine), len(ref))
    identical = bool(np.array_equal(mine[:n], ref[:n]))
    check3.append(identical)

    print("prompt:   ", repr(p))
    print("mine:     ", repr(tok.decode(mine[P:])))
    print("hf:       ", repr(tok.decode(ref[P:])))
    print("identical:", identical, "\n")

results["generation"] = all(check3)
print("PASS" if results["generation"] else "FAIL")

## Summary

In [ ]:
for name, ok in results.items():
    print(f"{name:12} {'PASS' if ok else 'FAIL'}")